# 凯撒密码 (Caesar Cipher) — 自学课程

**分类**：古典密码学

**内容简介**：介绍凯撒密码的原理、历史及实现方法。



## 学习目标

- 理解算法规则与数学表达
- 实现加解密函数，并写基本测试
- 通过小实验观察安全性与攻击方法
- 能解释常见踩坑并独立调试



## 前置知识与符号约定

- 字母表：仅使用大写 A-Z
- 映射：$A\rightarrow 0,\dots,Z\rightarrow 25$
- 模运算：$x\bmod 26$ 结果落在 $\{0,\dots,25\}$

> 如果你希望支持中文/标点，本课程也会在后续练习中给出扩展思路。



In [ ]:
# 课程通用工具：字母映射与规范化
import string
from collections import Counter, defaultdict
import math, random

ALPHABET = string.ascii_uppercase
A2I = {ch:i for i,ch in enumerate(ALPHABET)}
I2A = {i:ch for i,ch in enumerate(ALPHABET)}

def normalize(text: str) -> str:
    """仅保留 A-Z 并转大写"""
    return ''.join(ch for ch in text.upper() if ch in ALPHABET)

def keep_nonletters_encrypt(text: str, enc_func):
    """对字母加密，非字母原样保留（用于扩展练习）"""
    out=[]
    for ch in text:
        if ch.upper() in ALPHABET:
            out.append(enc_func(ch.upper()))
        else:
            out.append(ch)
    return ''.join(out)

print(normalize("Hello, World! 123"))
# 预期输出：HELLOWORLD



# Step 1：把算法写成数学模型

我们用统一抽象描述密码：

- 加密：$E_k: \mathcal{P}\to\mathcal{C}$
- 解密：$D_k: \mathcal{C}\to\mathcal{P}$
- 正确性：

$$D_k(E_k(p))=p$$

这能帮助你：
- 写出可测试的实现
- 分析密钥空间大小
- 讨论攻击模型（已知明文、选择明文等）



## 自检小测

1) 什么是“模 26”运算？为什么它会让结果回到 0..25？
2) 为什么实现中必须统一 A→0 的映射？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



> 常见错误 / 踩坑点 / 调试提示：
>
> - 不要混用 A→1 与 A→0 两种映射。
> - 记得对 k 做 k%=26；否则 k>26 时结果可能不符合预期。
> - 先写 round-trip 测试：decrypt(encrypt(p,k),k)==p。



# Step 2：凯撒密码规则与公式

凯撒密码是“移位替换”。设字母数值为 $x$，密钥为 $k$：

$$E_k(x)=(x+k)\bmod 26$$
$$D_k(y)=(y-k)\bmod 26$$

密钥空间大小：$|\mathcal{K}|=26$。



In [ ]:
# Step 2：凯撒加解密
def caesar_encrypt_text(pt: str, k: int) -> str:
    t=normalize(pt); k%=26
    return ''.join(I2A[(A2I[ch]+k)%26] for ch in t)

def caesar_decrypt_text(ct: str, k: int) -> str:
    t=normalize(ct); k%=26
    return ''.join(I2A[(A2I[ch]-k)%26] for ch in t)

print(caesar_encrypt_text("ATTACK AT DAWN", 3))
print(caesar_decrypt_text("DWWDFNDWGDZQ", 3))
# 预期输出：
# DWWDFNDWGDZQ
# ATTACKATDAWN



### 2.1 保留标点与空格（扩展实现）

真实文本里常有空格与标点。一个常见策略：

- 字母移位
- 非字母原样保留

这样更贴近日常使用，但也不会提升安全性。



In [ ]:
# 2.1：保留非字母字符的版本
def caesar_encrypt_keep(text: str, k: int) -> str:
    k%=26
    def enc_ch(ch):
        return I2A[(A2I[ch]+k)%26]
    return keep_nonletters_encrypt(text, enc_ch)

print(caesar_encrypt_keep("Hello, World!", 5))
# 预期输出：MJQQT, BTWQI!



# Step 3：最小测试（确保可逆）

写测试能让你快速发现：
- 映射是否一致
- k 是否取模
- 处理空字符串/边界情况是否正确



In [ ]:
# Step 3：轻量测试（assert）
def test_caesar():
    for k in range(0, 52):  # 覆盖多圈
        msg="THEQUICKBROWNFOXJUMPSOVERTHELAZYDOG"
        ct=caesar_encrypt_text(msg,k)
        pt=caesar_decrypt_text(ct,k)
        assert pt==msg
    assert caesar_encrypt_text("", 3)==""
    print("tests passed")

test_caesar()
# 预期输出：tests passed



# Step 4：攻击 1——暴力枚举

由于密钥只有 26 个，直接遍历所有 $k$ 就能得到所有候选明文。我们再用一个简单评分器挑前几名。



In [ ]:
# Step 4：评分器与爆破
COMMON = set("ETAOINSHRDLU")

def score_english_like(text: str) -> int:
    return sum(1 for ch in text if ch in COMMON)

def caesar_crack(ct: str, topk=5):
    ct=normalize(ct)
    cand=[]
    for k in range(26):
        pt=caesar_decrypt_text(ct,k)
        cand.append((score_english_like(pt), k, pt))
    cand.sort(reverse=True)
    return cand[:topk]

for s,k,pt in caesar_crack("DWWDFNDWGDZQ", topk=5):
    print(s, k, pt)
# 预期输出：应出现 k=3 且明文 ATTACKATDAWN



### 4.1 频率分析可视化（ASCII 直方图）

我们不依赖绘图库，用 ASCII 画一个简单直方图。频率分析对“替换密码”很重要。



In [ ]:
# 4.1：ASCII 直方图
def freq_hist(text: str):
    t=normalize(text)
    cnt=Counter(t)
    total=len(t) or 1
    for ch in ALPHABET:
        bar='█'*int(cnt[ch]/total*40)
        print(f"{ch}: {bar}")

sample = caesar_encrypt_text("THISISASIMPLEENGLISHTEXTFORFREQANALYSIS"*2, 7)
freq_hist(sample)
# 预期输出：打印 26 行直方图



# Step 5：攻击 2——已知明文与选择明文

- 已知明文：知道一对 $(p,c)$ 就能求 $k$。
- 选择明文：若能让系统加密 'A'，直接得到 $k$。

数学上：若 $p\rightarrow x$、$c\rightarrow y$，则

$$k\equiv y-x\pmod{26}$$



In [ ]:
# Step 5：从一对样本恢复 k
def recover_k_from_pair(p: str, c: str) -> int:
    p=normalize(p); c=normalize(c)
    if len(p)!=1 or len(c)!=1:
        raise ValueError("演示函数只处理单字母")
    return (A2I[c]-A2I[p])%26

k=recover_k_from_pair("A","D")
print(k)
# 预期输出：3



## 练习

1) 把评分器改成：统计常见二元组 TH/HE/IN/ER 出现次数。
2) 实现 `caesar_crack_keep(text)`：输入带标点的密文，输出带标点的候选明文。
3) 思考：为什么“扩大 k 的范围（比如 0..1000）”并不会提升安全性？


## 参考答案提示（可选）

- 二元组评分：遍历字符串的相邻对并计数。
- 带标点破解：可以只对字母部分爆破，再回填。

> 如果你卡住了：先让程序在短密文上工作，再逐步扩大到长文本。



## 总结与延伸

- 你已经完成：规则→数学→实现→测试→攻击/评估。
- 下一步可以：
  - 为实现添加更多字符集与格式化规则
  - 写更强的评分器做自动破译
  - 把多个古典密码组合，体验“组合不等于安全”

> 重要：古典密码主要用于学习思想；真实系统请使用经过标准化与审计的现代密码算法。



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”

